# Day 2 — Comprehensions, Functions, Scope
### Block B · Saint Mary's Hospital · Thursday morning

> *"Yesterday you saw the four data structures and the basics of functions. Today we **deepen** three things: comprehensions, function design, and scope. By lunch, you will be confident enough to write your own analysis from scratch."*

**Plan for today:**
- Morning (until ~12:45): we work through this notebook together. I show, you try.
- Afternoon: you work on the Exercise Notebook independently. Pure practice.

**By the end of today:**
1. You can transform a list of dicts into any other data structure with one comprehension.
2. You can write a function with parameters, defaults, and a clear return.
3. You can predict what variables exist where, and explain LEGB to a colleague.

---

> *"You won't learn a new library today. You'll get **fluent** in what you already know."*

## Setup — load yesterday's hospital data

In [1]:
# Run this once. We'll use 'patients' and 'encounters' throughout the morning.
patients = [
    {"id": "P0001", "age": 67, "gender": "F", "ward": "Cardiology",       "chronic": ["hypertension", "diabetes_t2"]},
    {"id": "P0002", "age": 54, "gender": "M", "ward": "InternalMedicine", "chronic": []},
    {"id": "P0003", "age": 82, "gender": "F", "ward": "Geriatrics",       "chronic": ["copd", "heart_failure", "ckd"]},
    {"id": "P0004", "age": 39, "gender": "M", "ward": "Surgery",          "chronic": []},
    {"id": "P0005", "age": 71, "gender": "F", "ward": "Cardiology",       "chronic": ["diabetes_t2"]},
    {"id": "P0006", "age": 90, "gender": "F", "ward": "Geriatrics",       "chronic": ["heart_failure", "ckd", "dementia"]},
    {"id": "P0007", "age": 58, "gender": "M", "ward": "Cardiology",       "chronic": ["obesity", "hypertension"]},
    {"id": "P0008", "age": 73, "gender": "M", "ward": "ICU",              "chronic": ["copd", "heart_failure"]},
]

encounters = [
    {"encounter_id": "E001", "patient_id": "P0001", "ward": "Cardiology",       "los_days": 5, "cost_eur": 3450.00},
    {"encounter_id": "E002", "patient_id": "P0003", "ward": "Geriatrics",       "los_days": 9, "cost_eur": 8950.00},
    {"encounter_id": "E003", "patient_id": "P0002", "ward": "InternalMedicine", "los_days": 3, "cost_eur": 1820.00},
    {"encounter_id": "E004", "patient_id": "P0005", "ward": "Cardiology",       "los_days": 4, "cost_eur": 2380.00},
    {"encounter_id": "E005", "patient_id": "P0008", "ward": "ICU",              "los_days": 7, "cost_eur": 7800.00},
    {"encounter_id": "E006", "patient_id": "P0006", "ward": "Geriatrics",       "los_days": 12, "cost_eur": 9840.00},
    {"encounter_id": "E007", "patient_id": "P0001", "ward": "Cardiology",       "los_days": 3, "cost_eur": 2110.00},
    {"encounter_id": "E008", "patient_id": "P0007", "ward": "Cardiology",       "los_days": 4, "cost_eur": 3950.00},
]

print(f"Loaded {len(patients)} patients and {len(encounters)} encounters.")

Loaded 8 patients and 8 encounters.


---

# Part 1 — Comprehensions, deeper

We saw comprehensions yesterday. Today we'll **manipulate** existing comprehensions to build intuition. Each block: I run it, you predict, then we change one thing and predict again.

## §1.1 — List comprehension, transform

The basic shape: `[expression for item in source]`.

In [2]:
# Get all ages
all_ages = [p["age"] for p in patients]
print(all_ages)

[67, 54, 82, 39, 71, 90, 58, 73]


**TRY THIS:** Change `p["age"]` to `p["ward"]`. What changes?

Then try `p["id"] + " (" + p["ward"] + ")"`. What kind of list do you get?

In [ ]:
# TRY: change the expression
all_X = [p["age"] for p in patients]   # ← change this
print(all_X)

## §1.2 — Filter

Add `if condition` at the end.

In [3]:
# Only patients aged 65+
elderly = [p["id"] for p in patients if p["age"] >= 65]
print(elderly)

['P0001', 'P0003', 'P0005', 'P0006', 'P0008']


**TRY THIS:**
- Change the threshold from 65 to 50.
- Then change the condition entirely: `p["ward"] == "Cardiology"`.
- Then combine both: `p["age"] >= 65 and p["ward"] == "Cardiology"` (don't forget parentheses-aren't-required-here-but-clear-writing-helps).

In [ ]:
# TRY: change the condition
result = [p["id"] for p in patients if p["age"] >= 65]
print(result)

## §1.3 — Multiple conditions

Logical operators inside the filter.

In [ ]:
# Cardiology AND age >= 65
cardio_seniors = [
    p["id"]
    for p in patients
    if p["ward"] == "Cardiology" and p["age"] >= 65
]
print(cardio_seniors)

# OR — patients with chronic conditions OR aged 80+
high_risk = [
    p["id"]
    for p in patients
    if len(p["chronic"]) > 0 or p["age"] >= 80
]
print(high_risk)

**Question to think about:** What's the difference between the two queries above? When would you use AND, when OR?

## §1.4 — Set comprehension — automatic deduplication

Same shape, **curly braces** instead of square brackets. Removes duplicates.

In [5]:
# All wards (with duplicates) — list comprehension
wards_list = [p["ward"] for p in patients]
print("With duplicates:", wards_list)

# All UNIQUE wards — set comprehension
wards_set = {p["ward"] for p in patients}
print("Unique:", wards_set)

With duplicates: ['Cardiology', 'InternalMedicine', 'Geriatrics', 'Surgery', 'Cardiology', 'Geriatrics', 'Cardiology', 'ICU']
Unique: {'Surgery', 'InternalMedicine', 'Cardiology', 'Geriatrics', 'ICU'}


**TRY THIS:** Make a set of all unique chronic conditions across all patients.

Hint: each patient has a LIST of conditions. You'll need a nested approach:
```python
{condition for p in patients for condition in p["chronic"]}
```
That's a comprehension with **two for-clauses** — the second iterates inside the first.

In [ ]:
# TRY: all unique chronic conditions
all_conditions = {condition for p in patients for condition in p["chronic"]}
print(all_conditions)

## §1.5 — Dict comprehension — build a lookup

Curly braces with a colon: `{key: value for ...}`.

In [4]:
# Patient ID → age lookup
age_by_id = {p["id"]: p["age"] for p in patients}
print(age_by_id)
print()
print("Age of P0003:", age_by_id["P0003"])

{'P0001': 67, 'P0002': 54, 'P0003': 82, 'P0004': 39, 'P0005': 71, 'P0006': 90, 'P0007': 58, 'P0008': 73}

Age of P0003: 82


**Why is this useful?** Once you have this dict, looking up any patient's age is **O(1)** — instant, regardless of dataset size. With a list of dicts, you'd have to scan everyone.

**TRY THIS:** Build a dict that maps patient ID to ward.

In [ ]:
# TRY: ID → ward
ward_by_id = {p["id"]: p["ward"] for p in patients}
print(ward_by_id)
print("Ward of P0003:", ward_by_id["P0003"])

## §1.6 — Comprehension with a transform

The output isn't just a field — it can be a calculation.

In [6]:
# Years until retirement (assume retirement age 67)
years_until_retire = {
    p["id"]: max(0, 67 - p["age"])
    for p in patients
}
print(years_until_retire)

{'P0001': 0, 'P0002': 13, 'P0003': 0, 'P0004': 28, 'P0005': 0, 'P0006': 0, 'P0007': 9, 'P0008': 0}


**TRY THIS:** Build a dict that maps patient ID to a string like `"P0001 (67y, F)"`. Use an f-string in the value.

In [ ]:
# TRY: ID → formatted summary
summary = {p["id"]: f"{p['id']} ({p['age']}y, {p['gender']})" for p in patients}
print(summary)

---

## §1.7 — When NOT to use a comprehension

Sometimes a regular for-loop is better. **The rule of thumb:** if you can't read it on one line without thinking, use a loop.

**Anti-example — too much logic in a comprehension:**

```python
# Don't do this — unreadable!
result = [
    f"{p['id']}: {' & '.join(p['chronic']) if p['chronic'] else 'no comorbidities'}"
    for p in patients
    if p["age"] >= 65 and (p["ward"] in ["Cardiology", "Geriatrics"] or len(p["chronic"]) >= 2)
]
```

**Better — split into a loop with named steps:**

In [7]:
# Readable version
result = []
for p in patients:
    is_old = p["age"] >= 65
    is_relevant_ward = p["ward"] in ["Cardiology", "Geriatrics"]
    has_multiple_conditions = len(p["chronic"]) >= 2

    if is_old and (is_relevant_ward or has_multiple_conditions):
        if p["chronic"]:
            description = " & ".join(p["chronic"])
        else:
            description = "no comorbidities"
        result.append(f"{p['id']}: {description}")

for line in result:
    print(line)

P0001: hypertension & diabetes_t2
P0003: copd & heart_failure & ckd
P0005: diabetes_t2
P0006: heart_failure & ckd & dementia
P0008: copd & heart_failure


---

# Part 2 — Functions, deeper

We saw `def` yesterday. Today we look at **how to design** a function: parameters, defaults, returns. The hospital case keeps growing.

## §2.1 — Recap: a function with one job

In [ ]:
def average_age(patient_list):
    """Return the mean age of a list of patients."""
    if not patient_list:
        return 0
    ages = [p["age"] for p in patient_list]
    return sum(ages) / len(ages)

# Use it
print("All patients:", average_age(patients))

# Use it on a subset — without changing the function!
cardio = [p for p in patients if p["ward"] == "Cardiology"]
print("Cardiology only:", average_age(cardio))

**The key insight**: the function doesn't know or care how the list was built. You can pass any subset to it. That's reusability.

## §2.2 — Default arguments

A parameter with a fallback value, used when the caller doesn't provide one.

In [ ]:
def filter_by_age(patient_list, min_age=65):
    """Return patients aged at least min_age."""
    return [p for p in patient_list if p["age"] >= min_age]

# Use the default
seniors = filter_by_age(patients)
print("Default (>=65):", [p["id"] for p in seniors])

# Override the default
younger = filter_by_age(patients, min_age=50)
print("With min_age=50:", [p["id"] for p in younger])

# Even older
super_seniors = filter_by_age(patients, min_age=80)
print("With min_age=80:", [p["id"] for p in super_seniors])

**TRY THIS:** Add a second parameter `max_age=None` that, when given, also filters out patients older than `max_age`. Hint: you'll need an `if max_age is not None` check inside the function.

In [ ]:
def filter_by_age_v2(patient_list, min_age=65, max_age=None):
    """Filter patients by age range."""
    result = [p for p in patient_list if p["age"] >= min_age]
    if max_age is not None:
        result = [p for p in result if p["age"] <= max_age]
    return result

print("65-75:", [p["id"] for p in filter_by_age_v2(patients, 65, 75)])
print("just min:", [p["id"] for p in filter_by_age_v2(patients, 60)])

## §2.3 — Multiple return values (via tuple)

Python lets you return more than one value — they come back as a tuple.

In [ ]:
def age_stats(patient_list):
    """Return (min, max, mean) age across a list of patients."""
    ages = [p["age"] for p in patient_list]
    return min(ages), max(ages), sum(ages) / len(ages)

# Capture all three with tuple unpacking
youngest, oldest, average = age_stats(patients)
print(f"Youngest: {youngest}, Oldest: {oldest}, Average: {average:.1f}")

# Or capture as one tuple if you don't need them separately
stats = age_stats(patients)
print("As tuple:", stats)

**TRY THIS:** Write a function `gender_breakdown(patient_list)` that returns `(n_female, n_male)`. Then call it and print the result with f-strings.

In [ ]:
def gender_breakdown(patient_list):
    """Return (count_female, count_male)."""
    n_female = sum(1 for p in patient_list if p["gender"] == "F")
    n_male = sum(1 for p in patient_list if p["gender"] == "M")
    return n_female, n_male

f_count, m_count = gender_breakdown(patients)
print(f"Female: {f_count}, Male: {m_count}")

## §2.4 — Functions calling other functions

The real power: build small functions, combine them.

In [ ]:
def is_high_risk(patient, age_threshold=65, min_conditions=2):
    """True if patient is old AND has multiple chronic conditions."""
    return p_age(patient) >= age_threshold and len(patient["chronic"]) >= min_conditions

def p_age(patient):
    """Helper: just get a patient's age."""
    return patient["age"]

def high_risk_patients(patient_list):
    """Return only high-risk patients."""
    return [p for p in patient_list if is_high_risk(p)]

# Use the chain
hr = high_risk_patients(patients)
print("High-risk patients:")
for p in hr:
    print(f"  {p['id']}: age {p['age']}, conditions: {p['chronic']}")

**Notice:** if tomorrow we change the rule for "high-risk" (e.g. include all patients aged 80+ regardless of conditions), we change **one function**. Everything else keeps working. This is what *modular* means.

## §2.5 — Lambda — for tiny one-shot functions

`lambda` is a one-line, anonymous function. Use it when you need a function for **just one operation**, typically as an argument to `sorted()`, `max()`, `min()`, or pandas methods.

In [8]:
# Sort patients by age, ascending
by_age = sorted(patients, key=lambda p: p["age"])
for p in by_age:
    print(f"{p['id']}: {p['age']}")

# Find the patient with the most chronic conditions
sickest = max(patients, key=lambda p: len(p["chronic"]))
print(f"\nMost conditions: {sickest['id']} with {len(sickest['chronic'])}")

P0004: 39
P0002: 54
P0007: 58
P0001: 67
P0005: 71
P0008: 73
P0003: 82
P0006: 90

Most conditions: P0003 with 3


**TRY THIS:** Sort patients by ward (alphabetical), then by age (descending) within ward. Hint: a tuple as key — `key=lambda p: (p["ward"], -p["age"])`. The minus reverses just the age.

In [15]:
# TRY: sort by ward then by age desc
by_ward_age = sorted(patients, key=lambda p: (p["ward"], -p["age"]))
for p in by_ward_age:
    print(f"{p['ward']:>40s}  {p['age']}  {p['id']}")

                              Cardiology  71  P0005
                              Cardiology  67  P0001
                              Cardiology  58  P0007
                              Geriatrics  90  P0006
                              Geriatrics  82  P0003
                                     ICU  73  P0008
                        InternalMedicine  54  P0002
                                 Surgery  39  P0004


---

# Part 3 — Scope, deeper

Yesterday's LEGB rule. Today: see it in action with concrete examples and a few traps.

## §3.1 — Local stays local

In [ ]:
def compute_score(patient):
    bonus = 10                                 # local to this function
    score = patient["age"] + len(patient["chronic"]) * 5 + bonus
    return score

risk = compute_score(patients[0])
print("Risk score:", risk)

# TRY: uncomment the next line and run again — what happens?
# print("Bonus:", bonus)

**Why does it crash?** Because `bonus` was a **local variable** inside `compute_score`. When the function returned, `bonus` was destroyed. Outside the function, no `bonus` exists.

## §3.2 — The LEGB walk

Three nested scopes. Watch which `x` Python finds at each `print`.

In [ ]:
x = 100                              # Module-level (Global)

def outer():
    x = 50                           # Local to outer (Enclosing for inner)

    def inner():
        x = 5                        # Local to inner
        print("Inside inner:", x)    # 5  — Local found first

    inner()
    print("Inside outer:", x)        # 50  — outer's local x

outer()
print("Module level:", x)            # 100  — global x

**Read it aloud:** *"Python looks up `x` using LEGB. It searches Local first, then Enclosing function (if any), then Global, then Built-in. Whoever is found first wins."*

## §3.3 — A common scope confusion

When students write this and wonder why their list is empty afterwards:

In [ ]:
def add_patient_id(target_list, p):
    """Add p's ID to target_list — looks like it should mutate."""
    target_list.append(p["id"])
    return target_list

# Approach A — assign the return:
collected = []
for p in patients[:3]:
    collected = add_patient_id(collected, p)
print("Approach A:", collected)

# Approach B — DON'T assign, just call (because lists are mutable, this also works)
collected2 = []
for p in patients[:3]:
    add_patient_id(collected2, p)
print("Approach B:", collected2)

Both work — because lists are mutable, the function changes the same list-object the caller is referring to. **But there's a trap:**

In [ ]:
def add_safely(target_list, value):
    """Looks safe, but reassigns inside — does it propagate outside?"""
    target_list = target_list + [value]   # reassignment, NOT mutation
    return target_list

# Without using the return value
my_list = []
add_safely(my_list, "P0001")
print("After call:", my_list)   # still empty!

# With using the return value
my_list = add_safely(my_list, "P0001")
print("After reassignment:", my_list)

**The lesson:** if a function uses `=` (reassignment) inside, it creates a new local object. The outer variable doesn't see it unless you `return` it AND the caller assigns. **Always be explicit about your contract: 'I mutate' vs 'I return a new copy'.**

## §3.4 — Two safe rules for the start

| Rule | Why |
|---|---|
| **Always `return` what you compute.** Don't rely on side effects. | Predictable, easy to test, no surprises. |
| **Don't read or write global variables from inside a function.** Pass them as parameters. | Functions become self-contained units. |

Follow these two and 80% of scope-related bugs disappear.

---

# Wrap-up — the three patterns that come up over and over

You can write almost any data analysis with these three building blocks:

```python
# 1. Filter (comprehension)
subset = [item for item in collection if condition]

# 2. Transform (comprehension)
new_collection = [f(item) for item in collection]

# 3. Aggregate (function)
def summary(collection):
    return some_metric_calculated_from(collection)
```

Combine them: filter → transform → aggregate. That's the workflow of every analysis you'll do for the rest of the course.

---

## This afternoon

Open `exercises_thursday.ipynb`. Eighteen exercises, three difficulty levels (basic / intermediate / fast-track). Work alone or in tandems. I'm here for questions.

The goal is **not** to finish all eighteen. The goal is **fluency** — that comprehensions, functions, and scope feel natural by the end of the day.

Tomorrow we'll add NumPy and Pandas — but only because you'll be ready for them by then.

---

> *"Today is the day you stop reading code and start writing it."*